# Prometheus Demonstration
End-to-end example workflow:
1. Load thermodynamic databases.
2. Define a combustion chamber equilibrium problem (HP).
3. Solve with both `HybridSolver` and `GordonMcBrideSolver`.
4. Compare key chamber metrics.
5. Run nozzle performance in frozen and shifting modes.

In [8]:
from __future__ import annotations

from pathlib import Path
from pprint import pprint
import time

from loguru import logger

from prometheus.equilibrium import (
    EquilibriumProblem,
    GordonMcBrideSolver,
    PerformanceSolver,
    ProblemType,
    SpeciesDatabase,
)


def _database_paths(repo_root: Path) -> dict[str, str]:
    """Return thermo database paths rooted at the local repository."""
    thermo_root = repo_root / "prometheus" / "thermo_data"
    return {
        "nasa7_path": str(thermo_root / "nasa7.json"),
        "nasa9_path": str(thermo_root / "nasa9.json"),
        "janaf_path": str(thermo_root / "janaf.csv"),
        "afcesic_path": str(thermo_root / "afcesic.json"),
        "terra_path": str(thermo_root / "terra.json"),
    }


REPO_ROOT = Path.cwd()
DB_PATHS = _database_paths(REPO_ROOT)
pprint(DB_PATHS)

# Keep notebook output readable even if a debug logger is configured globally.
logger.remove()


{'afcesic_path': 'C:\\Users\\1char\\Documents\\GitHub\\Prometheus\\prometheus\\thermo_data\\afcesic.json',
 'janaf_path': 'C:\\Users\\1char\\Documents\\GitHub\\Prometheus\\prometheus\\thermo_data\\janaf.csv',
 'nasa7_path': 'C:\\Users\\1char\\Documents\\GitHub\\Prometheus\\prometheus\\thermo_data\\nasa7.json',
 'nasa9_path': 'C:\\Users\\1char\\Documents\\GitHub\\Prometheus\\prometheus\\thermo_data\\nasa9.json',
 'terra_path': 'C:\\Users\\1char\\Documents\\GitHub\\Prometheus\\prometheus\\thermo_data\\terra.json'}


## 1) Load Thermodynamic Databases

In [9]:
db = SpeciesDatabase(**DB_PATHS)
db.load()

print(f"Loaded species count: {len(db.species)}")
print("Example lookup IDs:", sorted(list(db.species.keys()))[:5])


Loaded species count: 4559
Example lookup IDs: ['-_G', 'Ag+_G', 'Ag-_G', 'Ag2O_S', 'Ag2S_S']


## 2) Build an HP Chamber Problem (H2 / O2)

We solve an adiabatic constant-pressure chamber state:

- Fuel: H2(g)
- Oxidizer: O2(g)
- Initial reactants at 298.15 K
- Chamber pressure: 30 bar

In [10]:
sp_h2 = db.find("H2", phase="G")
sp_o2 = db.find("O2", phase="G")

# Stoichiometric H2/O2 for demonstration (2 H2 + 1 O2)
reactants = {sp_h2: 2.0, sp_o2: 1.0}

# Candidate products restricted to species that only use H/O elements
products = db.get_species({"H", "O"}, max_atoms=4)

T_react = 298.15
H0 = sum(n * sp.enthalpy(T_react) for sp, n in reactants.items())
P_chamber = 30.0e5  # 30 bar in Pa

hp_problem = EquilibriumProblem(
    reactants=reactants,
    products=products,
    problem_type=ProblemType.HP,
    constraint1=H0,
    constraint2=P_chamber,
    t_init=3500.0,
)

print(f"Candidate product species: {len(products)}")


Candidate product species: 39


In [11]:
# ## 3) Solve Chamber Equilibrium with Two Solvers


In [12]:
gmcb = GordonMcBrideSolver(capture_history=True)

start = time.perf_counter()
sol_gmcb = gmcb.solve(hp_problem)
t_gmcb = time.perf_counter() - start

print(
    f"G-McB converged={sol_gmcb.converged}, iters={sol_gmcb.iterations}, "
    f"T={sol_gmcb.temperature:.2f} K, runtime={1e3*t_gmcb:.2f} ms"
)


G-McB converged=True, iters=11, T=3549.71 K, runtime=6.15 ms


## 4) Chamber Composition Summary

In [13]:
pprint(sol_gmcb.major_species())

{'H': 0.04362472397574349,
 'H2': 0.1307275231638914,
 'H2O': 0.6562328090343089,
 'HO': 0.11081170675084431,
 'HO2': 0.0001694211321054247,
 'O': 0.01996309486810173,
 'O2': 0.038445614468514945}


## 5) Nozzle Performance (Frozen vs Shifting)

Use the converged chamber problem definition and solve to a fixed exit pressure.

In [14]:
perf = PerformanceSolver(solver=GordonMcBrideSolver(capture_history=False))

res_frozen = perf.solve(hp_problem, pe_pa=1.0e5, shifting=False)
print("Frozen:")
print(f"  c*      = {res_frozen.cstar:.2f} m/s")
print(f"  Isp_vac = {res_frozen.isp_vac:.2f} s")
print(f"  Ae/At   = {res_frozen.area_ratio:.3f}")

try:
    res_shifting = perf.solve(hp_problem, pe_pa=1.0e5, shifting=True)
    print("Shifting:")
    print(f"  c*      = {res_shifting.cstar:.2f} m/s")
    print(f"  Isp_vac = {res_shifting.isp_vac:.2f} s")
    print(f"  Ae/At   = {res_shifting.area_ratio:.3f}")
except Exception as exc:
    print(f"Shifting solve did not complete in this run: {exc}")

Frozen:
  c*      = 2103.83 m/s
  Isp_vac = 240.83 s
  Ae/At   = 14.032
Shifting:
  c*      = 2052.61 m/s
  Isp_vac = 279.80 s
  Ae/At   = 5.707


## 6) Quick Notes

- This demo is intentionally compact and deterministic.
- For broader validation against RocketCEA, run `tests/benchmark.py`.
- For API details, see `docs/` and module docstrings.
